In [1]:
# Angka tetap. Setiap undian acak di notebook ini memakai SEED.
SEED = 42


# Modul 9: Dasar jaringan saraf dan PyTorch

Cerita lengkapnya ada di `README.md`. Hitung dulu, baru jalankan kode.

Mode cepat memakai numpy di CPU. Nama langkahnya mengikuti PyTorch. Tidak ada unduhan. Bukan data asli perusahaan.

Dua hitungan tangan, sebelum ada jaringan:

1. Pada y = x * x dan x = 4, gradiennya berapa?
2. ReLU pada angka -1 menghasilkan berapa?


Sel akhir menjalankan torch di CPU hanya bila `A2N_TORCH=1` dan torch terpasang. Jika tidak, sel mencetak kalimat lewati.


In [2]:
# Satu ide: panggil numpy untuk menaruh angka dalam tensor.
import numpy as np


In [3]:
# Satu ide: tensor kecil menyimpan tiga angka dan punya bentuk.
tensor = np.array([1.0, 2.0, 3.0])
print("bentuk", tuple(tensor.shape))
print("jumlah", float(tensor.sum()))


bentuk (3,)
jumlah 6.0


In [4]:
# Satu ide: gradien tangan untuk y = x * x pada x = 4 adalah 2 * x.
x_nilai = 4.0
gradien_tangan = 2.0 * x_nilai
print("gradien_tangan", gradien_tangan)


gradien_tangan 8.0


In [5]:
# Satu ide: autograd mengingat perkalian agar gradien bisa diisi mundur.
class Angka:
    def __init__(self, nilai):
        self.nilai = float(nilai)
        self.grad = 0.0
        self._kiri = None
        self._kanan = None

    def __mul__(self, lain):
        hasil = Angka(self.nilai * lain.nilai)
        hasil._kiri = self
        hasil._kanan = lain
        return hasil


In [6]:
# Satu ide: langkah mundur mengisi gradien dari hasil ke kedua sisi kali.
def mundur(hasil):
    hasil.grad = 1.0
    hasil._kiri.grad += hasil.grad * hasil._kanan.nilai
    hasil._kanan.grad += hasil.grad * hasil._kiri.nilai

x = Angka(x_nilai)
y = x * x
mundur(y)
selisih_gradien = abs(x.grad - gradien_tangan)
print("gradien_autograd", x.grad)
print("selisih_gradien", selisih_gradien)
assert selisih_gradien < 1e-5


gradien_autograd 8.0
selisih_gradien 0.0


In [7]:
# Satu ide: fungsi aktivasi ReLU mematikan angka negatif.
def relu(z):
    return np.maximum(z, 0.0)

aktif = relu(np.array([-1.0, 0.0, 2.0]))
print("relu", aktif.tolist())


relu [0.0, 0.0, 2.0]


In [8]:
# Satu ide: nama yang sama di PyTorch, tanpa memasang paket torch.
print("nama_tensor", "torch.tensor")
print("nama_mundur", "backward")
print("nama_relu", "torch.nn.ReLU")


nama_tensor torch.tensor
nama_mundur backward
nama_relu torch.nn.ReLU


## Kasus nyata

Tabel bunga iris sudah ikut terpasang. Dua spesies yang dipakai sama dengan model klasifikasi Fase 1: versicolor dan virginica. Bukan data asli perusahaan.


In [9]:
# Satu ide: muat iris yang sudah terpasang, tanpa unduhan.
from sklearn.datasets import load_iris
import pandas as pd

iris = load_iris()
tabel = pd.DataFrame(iris.data, columns=iris.feature_names)
tabel["spesies"] = iris.target_names[iris.target]


In [10]:
# Satu ide: sisakan dua spesies yang sama dengan model Fase 1.
dua = tabel.loc[tabel["spesies"] != "setosa"].copy()
fitur = list(iris.feature_names)
print("baris", int(len(dua)))
print("spesies", sorted(dua["spesies"].unique().tolist()))


baris 100
spesies ['versicolor', 'virginica']


In [11]:
# Satu ide: pecah data latih dan data uji dengan SEED.
from sklearn.model_selection import train_test_split

latih, uji = train_test_split(
    dua, test_size=0.25, random_state=SEED, stratify=dua["spesies"]
)
print("baris_latih", int(len(latih)))
print("baris_uji", int(len(uji)))


baris_latih 75
baris_uji 25


In [12]:
# Satu ide: geser petunjuk memakai rata-rata data latih saja.
rerata = latih[fitur].to_numpy().mean(axis=0)
sebar = latih[fitur].to_numpy().std(axis=0)
X_latih = (latih[fitur].to_numpy() - rerata) / sebar
X_uji = (uji[fitur].to_numpy() - rerata) / sebar
y_latih = (latih["spesies"] == "virginica").to_numpy(dtype=float)
y_uji = (uji["spesies"] == "virginica").to_numpy(dtype=float)
print("petunjuk", int(X_latih.shape[1]))


petunjuk 4


In [13]:
# Satu ide: siapkan bobot kecil untuk lapisan di antara masukan dan keluaran.
rng = np.random.default_rng(SEED)
W1 = rng.normal(scale=0.1, size=(4, 4))
b1 = np.zeros(4)
w2 = rng.normal(scale=0.1, size=(4,))
b2 = 0.0
laju = 0.5
print("bentuk_W1", tuple(W1.shape))


bentuk_W1 (4, 4)


In [14]:
# Satu ide: delapan puluh putaran latihan memperkecil fungsi kerugian.
kerugian_awal = None
for _ in range(80):
    z1 = X_latih @ W1 + b1
    h = relu(z1)
    pred = 1 / (1 + np.exp(-(h @ w2 + b2)))
    if kerugian_awal is None:
        kerugian_awal = float(np.mean((pred - y_latih) ** 2))
    dz2 = (2 * (pred - y_latih) / len(y_latih)) * pred * (1 - pred)
    dz1 = (dz2[:, None] * w2) * (z1 > 0)
    W1, b1 = W1 - laju * (X_latih.T @ dz1), b1 - laju * dz1.sum(axis=0)
    w2, b2 = w2 - laju * (h.T @ dz2), b2 - laju * float(dz2.sum())
kerugian_akhir = float(np.mean((pred - y_latih) ** 2))


In [15]:
# Satu ide: fungsi kerugian pada data latih turun minimal 0.15.
turun = kerugian_awal - kerugian_akhir
print("kerugian_awal", round(kerugian_awal, 6))
print("kerugian_akhir", round(kerugian_akhir, 6))
print("turun", round(turun, 6))
assert turun >= 0.15


kerugian_awal 0.247573
kerugian_akhir 0.0296
turun 0.217973


In [16]:
# Satu ide: hitung tebakan jaringan yang salah pada data uji.
h_uji = relu(X_uji @ W1 + b1)
pred_uji = 1 / (1 + np.exp(-(h_uji @ w2 + b2)))
salah_uji_jaringan = int(((pred_uji >= 0.5) != (y_uji >= 0.5)).sum())
print("salah_uji_jaringan", salah_uji_jaringan)


salah_uji_jaringan 3


In [17]:
# Satu ide: bandingkan dengan baseline dan regresi logistik pada data yang sama.
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

baseline = DummyClassifier(strategy="most_frequent", random_state=SEED)
baseline.fit(X_latih, y_latih)
logistik = LogisticRegression(random_state=SEED)
logistik.fit(X_latih, y_latih)
salah_uji_baseline = int((baseline.predict(X_uji) != y_uji).sum())
salah_uji_logistik = int((logistik.predict(X_uji) != y_uji).sum())
print("salah_uji_baseline", salah_uji_baseline)
print("salah_uji_logistik", salah_uji_logistik)
assert salah_uji_jaringan < salah_uji_baseline


salah_uji_baseline 13
salah_uji_logistik 3


## Jalur PyTorch nyata

Sel berikut memakai `torch.tensor`, `backward`, dan jaringan kecil pada data iris yang sama. Semuanya di CPU. Sel jalan bila `A2N_TORCH=1` dan grup torch sudah terpasang. Jika tidak, sel mencetak kalimat lewati.


In [18]:
# Satu ide: jalur torch nyala hanya bila A2N_TORCH=1 dan torch terpasang.
import importlib.util
import os
ada_torch = importlib.util.find_spec("torch") is not None
jalur_torch = os.environ.get("A2N_TORCH") == "1" and ada_torch
PESAN = "Jalur ini dilewati. Pasang grup dengan uv sync --frozen --group torch, lalu set A2N_TORCH=1."
print("jalur_torch", "nyala" if jalur_torch else "lewati")


jalur_torch lewati


In [19]:
# Satu ide: torch.tensor menyimpan tiga angka yang sama dengan mode cepat.
if not jalur_torch:
    print("lewati_tensor", PESAN)
else:
    import torch
    torch.manual_seed(SEED)
    tensor_torch = torch.tensor([1.0, 2.0, 3.0])
    print("bentuk_torch", tuple(tensor_torch.shape))
    print("jumlah_torch", float(tensor_torch.sum()))


lewati_tensor Jalur ini dilewati. Pasang grup dengan uv sync --frozen --group torch, lalu set A2N_TORCH=1.


In [20]:
# Satu ide: backward mengisi gradien y = x * x pada x = 4.
if not jalur_torch:
    print("lewati_mundur", PESAN)
else:
    x_torch = torch.tensor(4.0, requires_grad=True)
    y_torch = x_torch * x_torch
    y_torch.backward()
    print("gradien_torch", float(x_torch.grad))
    assert abs(float(x_torch.grad) - 8.0) < 1e-5


lewati_mundur Jalur ini dilewati. Pasang grup dengan uv sync --frozen --group torch, lalu set A2N_TORCH=1.


In [21]:
# Satu ide: jaringan kecil torch memakai data iris yang sama, di CPU.
if not jalur_torch:
    print("lewati_jaringan", PESAN)
else:
    torch.manual_seed(SEED)
    jaring = torch.nn.Sequential(
        torch.nn.Linear(4, 4),
        torch.nn.ReLU(),
        torch.nn.Linear(4, 1),
    )
    print("cpu_torch", str(next(jaring.parameters()).device))


lewati_jaringan Jalur ini dilewati. Pasang grup dengan uv sync --frozen --group torch, lalu set A2N_TORCH=1.


In [22]:
# Satu ide: ubah petunjuk iris yang sudah digeser menjadi tensor CPU.
if not jalur_torch:
    print("lewati_data", PESAN)
else:
    xb = torch.tensor(X_latih, dtype=torch.float32)
    yb = torch.tensor(y_latih, dtype=torch.float32).view(-1, 1)
    opt = torch.optim.SGD(jaring.parameters(), lr=0.5)
    print("baris_torch", int(xb.shape[0]))


lewati_data Jalur ini dilewati. Pasang grup dengan uv sync --frozen --group torch, lalu set A2N_TORCH=1.


In [23]:
# Satu ide: delapan puluh backward menurunkan fungsi kerugian torch.
if not jalur_torch:
    print("lewati_latih", PESAN)
else:
    awal_torch = None
    for _ in range(80):
        opt.zero_grad()
        pred_t = torch.sigmoid(jaring(xb))
        loss_t = torch.mean((pred_t - yb) ** 2)
        if awal_torch is None:
            awal_torch = float(loss_t.detach())
        loss_t.backward()
        opt.step()


lewati_latih Jalur ini dilewati. Pasang grup dengan uv sync --frozen --group torch, lalu set A2N_TORCH=1.


In [24]:
# Satu ide: fungsi kerugian torch turun, dan salah uji lebih kecil dari baseline.
if not jalur_torch:
    print("lewati_hasil", PESAN)
else:
    turun_torch = awal_torch - float(loss_t.detach())
    xu = torch.tensor(X_uji, dtype=torch.float32)
    yu = torch.tensor(y_uji, dtype=torch.float32)
    with torch.no_grad():
        tebakan = torch.sigmoid(jaring(xu)).view(-1)
    salah_torch = int(((tebakan >= 0.5) != (yu >= 0.5)).sum())
    print("turun_torch", "ya" if turun_torch > 0 else "tidak")
    print("uji_torch", "lebih_kecil" if salah_torch < salah_uji_baseline else "tidak")
    assert turun_torch > 0
    assert salah_torch < salah_uji_baseline


lewati_hasil Jalur ini dilewati. Pasang grup dengan uv sync --frozen --group torch, lalu set A2N_TORCH=1.
